# Saving and loading parameters

A parametrization has to survive leaving the process that built it. Here it goes out to a
file and comes back, and the schema that guards the boundary is exercised on frames it
should refuse.

Every serializable class in the package implements the same three things — `schema`,
`to_frame`, `from_frame` — and inherits the rest from
`unito26.lob.frames.FrameSerializable`:

| form | out | in |
| --- | --- | --- |
| DataFrame | `to_frame()` | `from_frame(frame)` |
| records | `to_records()` | `from_records(records)` |
| JSON text | `to_json(indent)` | `from_json(text)` |
| JSON file | `write_json(path, indent)` | `read_json(path)` |

The records are the storage form: one dict per frame row, keyed by schema column, holding
plain Python scalars with `None` for a null. That is what `unito26/lob/config.py` holds,
written out as a literal, which is why an example parametrization can be read in the
source rather than decoded from a string.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
import pandera.errors

from unito26.lob import config
from unito26.lob.hawkes import HawkesParams
from unito26.lob.simulate import MarkParams

## 1. One row: the mark parameters

`MarkParams` says how an event type becomes an order. It is a single row, so its frame,
its record list and its JSON are all one element long.

In [ ]:
marks = MarkParams(depth_decay=0.45, mean_log_size=4.0, sigma_log_size=0.8, lot=10)
print(marks)
marks.to_frame()

In [ ]:
print(marks.to_records())
print()
print(marks.to_json(2))

## 2. The schema is the boundary

`schema()` returns the pandera schema every frame form validates against, in both
directions: `to_frame` validates what it builds, and `from_frame` validates what it is
given before reading it. So a frame that is wrong is refused where it enters, rather than
producing a parametrization that is quietly outside its domain.

In [ ]:
schema = MarkParams.schema()
pd.DataFrame(
    [{"column": name, "dtype": str(column.dtype), "nullable": column.nullable,
      "checks": [str(check) for check in column.checks]}
     for name, column in schema.columns.items()]
).set_index("column")

In [ ]:
def refused(frame):
    try:
        MarkParams.from_frame(frame)
    except (pandera.errors.SchemaError, pandera.errors.SchemaErrors) as error:
        return f"refused ({type(error).__name__})"
    return "accepted"

off_domain = marks.to_frame()
off_domain.loc[0, "DepthDecay"] = 1.5          # a geometric parameter above 1
missing = marks.to_frame().drop(columns=["Lot"])
extra = marks.to_frame().assign(Comment="a note")

print("DepthDecay = 1.5 :", refused(off_domain))
print("no Lot column    :", refused(missing))
print("an extra column  :", refused(extra))
print("DepthDecay = 1.0 :", refused(marks.to_frame().assign(DepthDecay=1.0)))

`DepthDecay = 1` is accepted: it is the geometric parametrization that puts every order at
the touch, which is a choice and not an error. The domain is $(0, 1]$, and stating which
end is open is part of what the schema is for.

## 3. Many rows: the Hawkes parameters

`HawkesParams` holds a vector $\mu$, a matrix $A$ and a scalar $\beta$, and the frame is
their long form: one row per (excited, exciting) pair, with the baseline on the diagonal
and null off it, and the decay repeated.

`from_frame` **pivots** on `(Component, Cause)` rather than reshaping in row order. No
schema can constrain the order of rows, and a frame whose rows arrived shuffled would
reshape into the transpose — which is a different model, with the excitation running the
other way, and no error to say so.

In [ ]:
flow = HawkesParams(baseline=[0.3, 0.7], excitation=[[1.0, 2.0], [3.0, 4.0]], decay=60.0)
flow.to_frame().set_index(["Component", "Cause"])

In [ ]:
shuffled = flow.to_frame().sample(frac=1.0, random_state=0)
print("rows in this order:", [(int(i), int(j)) for i, j in
                             zip(shuffled["Component"], shuffled["Cause"])])
print("excitation recovered:")
print(HawkesParams.from_frame(shuffled).excitation)

## 4. Out to a file, and back

`write_json` and `read_json` are the pair to use for a parametrization that is meant to
outlive the session. The file is a JSON array of records: readable, diffable, and the same
shape a constant in `config.py` has.

In [ ]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "order-flow.json"
    config.example_order_flow_params().write_json(path, indent=2)

    print(path.read_text().splitlines()[:6])
    print("...")
    print(f"{path.stat().st_size} bytes")

    reloaded = HawkesParams.read_json(path)

original = config.example_order_flow_params()
print("\nbaseline equal   ", np.array_equal(reloaded.baseline, original.baseline))
print("excitation equal ", np.array_equal(reloaded.excitation, original.excitation))
print("decay equal      ", reloaded.decay == original.decay)
print("branching ratio  ", round(reloaded.branching_ratio, 6))

## 5. The frozen examples

`config` ships examples, not defaults. A default invites a reader to skip the choice, and
the choice is the subject; an example invites them to open it and see what a specification
looks like. Nothing there is computed at import time — each constant is a record list, and
the loaders are the only way in.

In [ ]:
print(config.EXAMPLE_MARK_PARAMS)
print()
print(config.EXAMPLE_ORDER_FLOW_PARAMS[:3], "...", f"({len(config.EXAMPLE_ORDER_FLOW_PARAMS)} rows)")
print()
print("the loader reproduces the constant:",
      config.example_order_flow_params().to_records() == config.EXAMPLE_ORDER_FLOW_PARAMS)